# IOAI — 2024 Summer National Sms Spam (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!pip -q install datasets
from datasets import load_dataset
import pandas as pd
ds = load_dataset('ucirvine/sms_spam')['train']
df = pd.DataFrame({'text': ds['sms'], 'label': ds['label']}).sample(frac=1, random_state=42).reset_index(drop=True)
cut = int(len(df) * 0.8); tr, te = df[:cut], df[cut:].reset_index(drop=True)
tr[['text','label']].to_csv('train.csv', index=False)
te.assign(id=range(len(te)))[['id','text']].to_csv('test.csv', index=False)
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# SMS Spam Detection (Spam SMS-ek)

> **HAIO 2024 — Summer National Finals (NLP)**

Classify each SMS as **spam (1)** or **ham (0)**. Score = **F1** (spam). `train.csv` is labelled; `test.csv` (id,text) is graded against held-out labels via **Submit** (`submission.csv` = `id,label`).

Baseline: TF-IDF + Logistic Regression.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
train=pd.read_csv('train.csv'); test=pd.read_csv('test.csv')
train['text']=train['text'].astype(str); test['text']=test['text'].astype(str)
train.shape, test.shape, train['label'].mean()

## Self-check (held-out split) then train on all + predict test

In [ ]:
Xtr,Xva,ytr,yva=train_test_split(train['text'],train['label'],test_size=0.2,random_state=42,stratify=train['label'])
vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True); clf=LogisticRegression(max_iter=1000,C=8.0,class_weight='balanced')
clf.fit(vec.fit_transform(Xtr),ytr)
print('val F1:',round(f1_score(yva,clf.predict(vec.transform(Xva))),4))

In [ ]:
vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True); clf=LogisticRegression(max_iter=1000,C=8.0,class_weight='balanced')
clf.fit(vec.fit_transform(train['text']),train['label'])
pred=clf.predict(vec.transform(test['text']))
pd.DataFrame({'id':test['id'],'label':pred}).to_csv('submission.csv',index=False); print('wrote',len(test))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)